#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import trim,col,length

#Reading From Bronze

In [0]:
df = spark.sql("""SELECT 
        sls_ord_num,
        sls_prd_key,
        sls_cust_id,
         CASE 
         WHEN sls_order_dt = 0 
             OR LENGTH(CAST(sls_order_dt AS STRING)) != 8
         THEN NULL
         ELSE TO_DATE(CAST(sls_order_dt AS STRING), 'yyyyMMdd')
         END AS sls_order_dt,
        CASE 
         WHEN sls_ship_dt = 0 
             OR LENGTH(CAST(sls_ship_dt AS STRING)) != 8
         THEN NULL
         ELSE TO_DATE(CAST(sls_ship_dt AS STRING), 'yyyyMMdd')
        END AS sls_ship_dt,
        CASE 
         WHEN sls_due_dt = 0 
             OR LENGTH(CAST(sls_due_dt AS STRING)) != 8
         THEN NULL
         ELSE TO_DATE(CAST(sls_due_dt AS STRING), 'yyyyMMdd')
        END AS sls_due_dt,
        sls_sales,
        sls_quantity,
        sls_price FROM bronze.crm_sales_details""")

#Data Transformation

##Triming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df=df.withColumn(field.name,F.trim(col(field.name)))

##Sales and Price Corrections

In [0]:
df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)

##Renaming Columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name,new_name in RENAME_MAP.items():
    df=df.withColumnRenamed(old_name,new_name)


#Sanity check of dataframe

In [0]:

df.limit(10).display()

#Write Into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.crm_sales")

#Sanity check of Silver Table

In [0]:
%sql
SELECT * FROM silver.crm_sales
LIMIT 10